# 🔍 Notebook 06 — SHAP: Interpretabilidade do Modelo XGBoost
## Predictfy × Locaweb — FIAP Challenge 2026

**Objetivo:** Explicar as previsões do modelo XGBoost usando SHAP values,
identificando quais features mais influenciam o risco de violação de OLA.

**Dependência:** Executar após o notebook 04 (modelo XGBoost treinado)
**Input:** Modelo salvo + `incidents_features.parquet`
**Output:** Análise visual + `outputs/shap_features.json`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import pickle
import warnings
warnings.filterwarnings('ignore')
import json, os
from datetime import date

print("✅ Imports ok")
shap.initjs()


In [ ]:
# ── Carregar modelo e dados ───────────────────────────────────────────────────
df = pd.read_parquet('../data/processed/incidents_features.parquet')

FEATURES = [
    'hora', 'dia_semana', 'mes', 'trimestre',
    'is_horario_comercial', 'is_fim_de_semana', 'is_segunda_terca',
    'periodo_dia', 'prioridade_bin',
    'lag_1d', 'lag_7d', 'rolling_7d', 'rolling_30d',
    'lag_1d_p2', 'lag_1d_p3',
    'Produto_enc', 'Categoria_enc', 'Grupo designado_enc',
    'Produto_freq', 'Grupo designado_freq',
]

X = df[FEATURES]

# Carregar modelo salvo do nb04
with open('../models_saved/xgboost_ola_risk.pkl', 'rb') as f:
    model = pickle.load(f)

print("✅ Modelo carregado")
print(f"Shape dos dados: {X.shape}")


In [ ]:
# ── Calcular SHAP values ──────────────────────────────────────────────────────
# Usar amostra para velocidade (SHAP é lento no dataset completo)
X_sample = X.sample(n=5000, random_state=42)

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

print(f"SHAP values shape: {shap_values.shape}")
print("✅ SHAP values calculados")


In [ ]:
# ── Summary Plot ─────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, plot_type='bar',
                  feature_names=FEATURES, show=False)
plt.title('SHAP Feature Importance — Risco de Violação OLA')
plt.tight_layout()
plt.savefig('../outputs/shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ shap_summary_bar.png salvo")


In [ ]:
# ── Beeswarm Plot ─────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURES, show=False)
plt.title('SHAP Beeswarm — Distribuição dos Valores por Feature')
plt.tight_layout()
plt.savefig('../outputs/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ shap_beeswarm.png salvo")


In [ ]:
# ── Top features por importância SHAP média ──────────────────────────────────
mean_shap = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({
    'feature': FEATURES,
    'shap_importance': mean_shap,
    'shap_importance_pct': mean_shap / mean_shap.sum() * 100
}).sort_values('shap_importance', ascending=False)

print("Top 10 features por SHAP importance:")
print(shap_df.head(10).to_string(index=False))

# Exportar
output = {
    "modelo": "xgboost_shap_interpretation",
    "gerado_em": date.today().strftime('%Y-%m-%d'),
    "features": shap_df[['feature','shap_importance','shap_importance_pct']].assign(
        shap_importance=lambda d: d['shap_importance'].round(4),
        shap_importance_pct=lambda d: d['shap_importance_pct'].round(2)
    ).to_dict('records'),
}

os.makedirs('../outputs', exist_ok=True)
with open('../outputs/shap_features.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("\n✅ shap_features.json exportado")
